# oxDNA: Running a Simulation

This notebook demonstrates how to run a basic oxDNA simulation using mythos and read back the trajectory output.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

In this notebook we run a simple oxDNA simulation: we set up the energy function and simulator, launch a simulation, and inspect the resulting trajectory. This is the minimal starting point for any oxDNA-based workflow in mythos.

## Imports

In [ ]:
from pathlib import Path

import mythos.energy.dna1 as dna1_energy
import mythos.input.topology as jdna_top
import mythos.simulators.oxdna as jdna_oxdna

## Setup

Point to the input directory that contains the oxDNA configuration files (`input`, `sys.top`, etc.). This directory ships with the repository under `data/templates/`.

In [ ]:
input_dir = Path("../../data/templates/simple-helix")
topology = jdna_top.from_oxdna_file(input_dir / "sys.top")

## Build the energy function

`create_default_energy_fn` assembles the full oxDNA1 composed energy function (FENE, stacking, hydrogen bonding, excluded volume, cross-stacking, coaxial stacking) from the default parameter sets for the given topology.

In [ ]:
energy_fn = dna1_energy.create_default_energy_fn(topology=topology)

## Create and run the simulator

The `oxDNASimulator` compiles and runs the oxDNA binary with the parameters encoded in the energy function. You need either a `source_path` (to compile from source) or a `binary_path` (to use a precompiled binary).

In [ ]:
simulator = jdna_oxdna.oxDNASimulator(
    input_dir=input_dir,
    energy_fn=energy_fn,
    source_path=Path("../../../oxDNA").resolve(),
)

output = simulator.run()

## Read the trajectory

After the simulation completes, we read the output trajectory and inspect its
shape. Each simulator "exposes" a set of named observables that are returned in
the output structure in the order given by `simulator.exposes()`. The
`oxDNASimulator` exposes the trajectory as its only observable. The naming is
relevant particularly in the case of multi-simulator/multi-objective workflows.

In [ ]:
simulator.exposes()

In [ ]:
trajectory = output.observables[0]  # The trajectory is the first (and only) observable exposed by the simulator

print("Length of trajectory:", trajectory.length())
# display the first 2 centers of the first two frames
print(trajectory.center[:2, :2, :])